# Spotify Recommender System – Part 3: Collaborative Filtering Models
**Author:** Miguel Vásquez  
**Date:** 23 September 2025  

---

## 1. Introduction  

In this notebook, we implement **collaborative filtering approaches** to capture user–item interaction patterns. Unlike the popularity baseline, these models leverage the co-occurrence of tracks across playlists to provide more personalized recommendations.  

We focus on two widely used methods:  
- **Alternating Least Squares (ALS)** for matrix factorization.  
- **LightFM** for hybrid recommendations combining collaborative and content-based features.  

---

## 2. Objectives of this notebook  

- Prepare the user–item interaction matrix.  
- Train ALS and LightFM models on the 150k playlist subset.  
- Generate personalized top-K recommendations.  
- Compare their performance against the baseline.  

In [1]:
# imports 
import os
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_palette("Set3")

# ensure working dir (if you run from notebooks/)
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
    display(Markdown(f"Changed working dir to {os.getcwd()}"))

Changed working dir to c:\Users\Miguel\portfolio\Spotify-Recommender-System

## 3. Data preparation
### 3.1 Load processed tracks dataset
Load the playlist and track subset from the previous notebook (`tracks_subset.parquet`).

In [2]:
tracks_path = Path("data/subset/tracks_subset.parquet")
assert tracks_path.exists(), f"File not found: {tracks_path}"
tracks_df = pd.read_parquet(tracks_path)

display(Markdown(f"**Loaded tracks_df** rows={len(tracks_df)}, unique playlists={tracks_df['playlist_id'].nunique()}"))
display(tracks_df.head())

**Loaded tracks_df** rows=9994184, unique playlists=150000

,playlist_id,artist_name,track_name
0,0,Missy Elliott,Lose Control (feat. Ciara & Fat Man Scoop)
1,0,Britney Spears,Toxic
2,0,Beyoncé,Crazy In Love
3,0,Justin Timberlake,Rock Your Body
4,0,Shaggy,It Wasn't Me


### 3.2 Define unique user–item keys
- Create stable `item_key` (`track_uri` or fallback with `track_name + artist_name`).
- Map `playlist_id → user_idx` and `item_key → item_idx`.

In [3]:
# create stable item_key for tracks
tracks_df['item_key'] = tracks_df['track_name'].str.strip() + " - " + tracks_df['artist_name'].str.strip()

# map playlist -> user_idx and item_key -> item_idx (integers)
playlist_ids = tracks_df['playlist_id'].unique()
item_keys = tracks_df['item_key'].unique()

user2idx = {pid: i for i, pid in enumerate(playlist_ids)}
item2idx = {it: i for i, it in enumerate(item_keys)}

tracks_df['user_idx'] = tracks_df['playlist_id'].map(user2idx)
tracks_df['item_idx'] = tracks_df['item_key'].map(item2idx)

n_users = len(user2idx)
n_items = len(item2idx)
display(Markdown(f"n_users={n_users}, n_items={n_items}"))

n_users=150000, n_items=831416

### 3.3 Build interaction matrix
- Construct a sparse matrix (CSR) of size n_users × n_items.
- Confirm dimensions and sparsity.

In [4]:
from scipy.sparse import coo_matrix, csr_matrix

# aggregate counts per (user,item) (a track may appear multiple times in same playlist)
inter_agg = tracks_df.groupby(['user_idx','item_idx']).size().reset_index(name='count')

rows = inter_agg['user_idx'].values
cols = inter_agg['item_idx'].values
data = inter_agg['count'].astype(float).values

interactions = coo_matrix((data, (rows, cols)), shape=(n_users, n_items)).tocsr()
display(Markdown(f"interactions shape {interactions.shape}, nnz={interactions.nnz}"))

interactions shape (150000, 831416), nnz=9853165

## 4. Train-Test Split
- Separate users (playlists) into train (80%) and test (20%).
- Build `train_interactions` and `test_interactions` matrices.

In [5]:
# Split playlists (users). Train used for model fitting; test for evaluation.
from sklearn.model_selection import train_test_split
rng = np.random.RandomState(42)

all_user_idxs = np.arange(n_users)
train_users, test_users = train_test_split(all_user_idxs, test_size=0.20, random_state=42)

# Build train and test interaction matrices
train_interactions = interactions[train_users, :]
test_interactions = interactions[test_users, :]

display(Markdown(f"train users: {train_interactions.shape[0]}, test users: {test_interactions.shape[0]}"))

train users: 120000, test users: 30000

## 5. Model Training
### 5.1 ALS (Matrix Factorization)
- Implement ALS with the `Implicit` library.
- Explain the concept of latent factorization.

In [ ]:
import pickle
from implicit.als import AlternatingLeastSquares

# ALS model training verification
os.makedirs("models", exist_ok=True)
als_path = "models/als_model.pkl"

if os.path.exists(als_path):
    display(Markdown("ALS model found. Loading from disk..."))
    with open(als_path, "rb") as f:
        als = pickle.load(f)
else:
    display(Markdown("Training ALS (this may take a while)..."))
    alpha = 40.0
    # Implicit expects item-user matrix (items x users) and scaling for confidence
    item_user = (train_interactions.T * alpha).astype('double')
    als = AlternatingLeastSquares(
        factors=64, 
        regularization=0.05, 
        iterations=15, 
        use_gpu=True
    )
    als.fit(item_user)
    with open(als_path, "wb") as f:
        pickle.dump(als, f)
    display(Markdown("ALS trained and saved."))

c:\Users\Miguel\portfolio\Spotify-Recommender-System\venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Miguel\portfolio\Spotify-Recommender-System\venv310\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Fitting ALS (this may take a while)...

c:\Users\Miguel\portfolio\Spotify-Recommender-System\venv310\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.09098386764526367 seconds
  warnings.warn(
100%|██████████| 15/15 [00:19<00:00,  1.32s/it]


ALS trained.

: 

### 5.2 LightFM (Hybrid Collaborative Filtering)
- Explain LightFM and how it combines interactions and features.
- Include simple `artist_name` as an item feature.
- Train with `loss="warp"`.

In [ ]:
import os, pickle
from lightfm import LightFM
from sklearn.feature_extraction.text import CountVectorizer

# LightFM model training with artist features verification
os.makedirs("models", exist_ok=True)

lfm_path = "models/lightfm_model.pkl"

if os.path.exists(lfm_path):
    display(Markdown("LightFM model found. Loading from disk..."))
    with open(lfm_path, "rb") as f:
        lfm = pickle.load(f)
else:
    display(Markdown("Preparing artist features..."))
    # Build item->artist mapping (ordered by item_idx)
    item_artist = (tracks_df[['item_idx','artist_name']]
                   .drop_duplicates(subset=['item_idx'])
                   .set_index('item_idx').reindex(range(n_items))['artist_name']
                   .fillna('unknown').astype(str))
    # vectorize artist names as sparse one-hot-like features
    artist_vec = CountVectorizer(token_pattern='[^,]+')
    artist_features = artist_vec.fit_transform(item_artist.values)

    display(Markdown(f"artist_features shape: {artist_features.shape}"))

    lfm = LightFM(no_components=64, loss='warp')
    display(Markdown("Training LightFM (this may take a while)..."))
    lfm.fit(train_interactions, item_features=artist_features, epochs=10, num_threads=-1)

    with open(lfm_path, "wb") as f:
        pickle.dump(lfm, f)
    display(Markdown("LightFM trained and saved."))


c:\Users\Miguel\portfolio\Spotify-Recommender-System\venv310\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


artist_features shape: (831416, 129076)

Training LightFM (this may take a while)...

## 6. Recommendation Generation
### 6.1 ALS Recommendations
- Function to recommend top-K items for a user.]


### 6.2 LightFM Recommendations
- Function to recommend top-K items with features.

### 6.3 Example outputs
- Show recommendations for some test users (e.g. 5 playlists).

## 7. Evaluation
### 7.1 Ground truth construction
- Define set of items per playlist in test set.

### 7.2 Metrics definition
- Implement **Precision@K, Recall@K, HitRate@K**.

### 7.3 Evaluation results
- Calculate metrics for both models at different K (5, 10, 20).
Present a comparative table of results.

## 8. Discussion
- Compare ALS vs. LightFM results. 
- Identify strengths and limitations of each approach.
- Contrast with the popularity baseline (from Notebook 02).

## 9. Next Steps
- Possible improvements: tuning with Optuna, additional features, dimensionality reduction.
- Link to the next notebook (04_evaluation) for more detailed analysis.